# HVAC Schedule Extractor - Kaggle Batch Run

Runs `dataextraction` over a whole corpus of drawing-set PDFs and produces the
metadata directory: `hvac.csv` (one row per equipment item) and `hvac_team.csv`.

**No GPU needed.** The structuring step is a Groq API call, so this is I/O bound -
set Accelerator to *None*. A GPU only matters if the OCR pass for vector-line-art
schedules is added later.

### Before you run
1. **Settings -> Internet -> On** (needed to clone and to reach the Groq API).
2. **Add-ons -> Secrets** -> add `GROQ_API_KEY` with your `gsk_...` key.
3. **Attach your PDF corpus** as a Kaggle Dataset input.
4. If resuming, also attach the previous run's output so `hvac.db` is restored -
   already-processed files are then skipped by content hash.


## 1. Environment


In [ ]:
import sys, platform, subprocess
print('python  :', sys.version.split()[0], platform.machine())
print('platform:', platform.platform())


## 2. Get the code

Two sources, in order of preference:
- clone from GitHub (set `BRANCH` - the fixes live on `merge/main-pranali-sahil`
  until it is merged to `main`);
- otherwise fall back to the code attached as a Kaggle Dataset, for when the branch
  has not been pushed yet.


In [ ]:
import os, glob, shutil

REPO_URL = 'https://github.com/triunesolutions/dataextraction'
BRANCH   = 'merge/main-pranali-sahil'   # change to 'main' once merged
WORK     = '/kaggle/working'
REPO_DIR = os.path.join(WORK, 'dataextraction')

# Step out of REPO_DIR before anything deletes it. Re-running this cell
# otherwise leaves the process inside a directory that no longer exists,
# and every subprocess then dies with 'Unable to read current working
# directory' -- which looks like a clone failure but is not one.
os.chdir(WORK)

def _try_clone():
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                        REPO_URL, REPO_DIR],
                       cwd=WORK, capture_output=True, text=True)
    if r.returncode != 0:
        print('clone failed:', r.stderr.strip()[-300:])
    return r.returncode == 0

def _try_dataset():
    # Any attached dataset holding run.py + extract.py will do.
    for cand in glob.glob('/kaggle/input/*/run.py') + glob.glob('/kaggle/input/*/*/run.py'):
        src = os.path.dirname(cand)
        if os.path.exists(os.path.join(src, 'extract.py')):
            shutil.copytree(src, REPO_DIR, dirs_exist_ok=True)
            print('using code from attached dataset:', src)
            return True
    return False

assert _try_clone() or _try_dataset(), (
    'No code source. Either push the branch, or attach the repo as a Kaggle Dataset.')
os.chdir(REPO_DIR)
print('code at', REPO_DIR)
print(subprocess.run(['git', 'log', '--oneline', '-1'], cwd=REPO_DIR,
                     capture_output=True, text=True).stdout.strip())


## 3. Dependencies

`pdfplumber` is what reconstructs a CAD table from its ruled grid. Without it the
extractor falls back to pypdf's scrambled paint-order text and quietly produces worse
manufacturer/model pairing, so the install is verified rather than assumed.


In [ ]:
!pip -q install -r requirements.txt

import importlib.util
missing = [m for m in ('pypdf', 'pdfplumber', 'pydantic', 'dotenv')
           if importlib.util.find_spec(m) is None]
assert not missing, f'missing dependencies: {missing}'
import pdfplumber, pypdf
print('pdfplumber', pdfplumber.__version__, '| pypdf', pypdf.__version__)


## 4. Credentials

Held in the environment rather than written to `.env` inside the repo. Cell 2 deletes
and re-clones that directory, which silently destroys a `.env` written here - so a
re-clone would leave the run with no key, failing well after the point that caused it.
Environment variables survive the re-clone and are inherited by the run in cell 6.

`config.py` reads `os.getenv` and `load_dotenv` does not override an existing variable,
so this takes precedence cleanly. The key is never printed.


In [ ]:
from kaggle_secrets import UserSecretsClient

key = UserSecretsClient().get_secret('GROQ_API_KEY')
assert key and key.startswith('gsk_'), 'GROQ_API_KEY secret missing or malformed'

os.environ['MODEL_BACKEND'] = 'groq'
os.environ['GROQ_API_KEY']  = key
os.environ['GROQ_MODEL']    = 'openai/gpt-oss-120b'
print('credentials set in environment (key not shown)')


## 5. Find the corpus, restore any previous database

`run.py` walks folders recursively, so `CORPUS` only needs to be the directory the
project folders sit under. Restoring `hvac.db` from a previous run is what makes the
12-hour session limit survivable: finished files are skipped by content hash, so the
next session resumes where this one stopped.


In [ ]:
CORPUS = None
for root in sorted(glob.glob('/kaggle/input/*')):
    if glob.glob(os.path.join(root, '**', '*.pdf'), recursive=True):
        CORPUS = root
        break
assert CORPUS, 'No attached dataset contains PDFs - attach your corpus.'

pdfs = glob.glob(os.path.join(CORPUS, '**', '*.pdf'), recursive=True)
print('CORPUS =', CORPUS)
print(len(pdfs), 'PDF(s) found')

DB = '/kaggle/working/hvac.db'
prior = (glob.glob('/kaggle/input/*/hvac.db') + glob.glob('/kaggle/input/*/*/hvac.db'))
if prior and not os.path.exists(DB):
    shutil.copy(prior[0], DB)
    print('restored previous database from', prior[0])
elif os.path.exists(DB):
    print('continuing this session database')
else:
    print('starting a fresh database')


## 6. Run the batch

Output streams live. Groq rate limits, not compute, are the bottleneck: the extractor
backs off on 429 and splits oversized requests on 413, but a large corpus still spends
most of its wall-clock waiting on the API.

Leave `LIMIT` small for a smoke test before committing to the whole corpus.


In [ ]:
LIMIT = 20   # 0 = process everything

cmd = [sys.executable, 'run.py', CORPUS, '--db', DB,
       '--export', '/kaggle/working/hvac.csv']
if LIMIT:
    cmd += ['--limit', str(LIMIT)]
print(' '.join(cmd), flush=True)

proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('exit code:', proc.returncode)


## 7. Quality report

The numbers that decide whether this corpus is usable. A file reporting `ok` with zero
rows is the failure mode that made an earlier 67-file run look clean while silently
dropping a third of it, so it is called out explicitly rather than left to be noticed.


In [ ]:
import sqlite3, pandas as pd

con = sqlite3.connect(DB)

print('=== status ===')
display(pd.read_sql('SELECT status, COUNT(*) n FROM files GROUP BY status', con))

rows = pd.read_sql('''
    SELECT f.filename, f.status,
           (SELECT COUNT(*) FROM equipment e WHERE e.file_hash = f.file_hash)
               AS equipment_rows,
           p.project_name, p.location, p.engineer,
           COALESCE(f.error, '') AS error
    FROM files f LEFT JOIN projects p ON p.file_hash = f.file_hash
''', con)

empty_ok = rows[(rows.status == 'ok') & (rows.equipment_rows == 0)]
print('=== files reporting ok with 0 equipment rows:', len(empty_ok), '===')
if len(empty_ok):
    display(empty_ok[['filename', 'error']])

n = max(len(rows), 1)
print('=== field coverage (per file) ===')
for col in ('project_name', 'location', 'engineer'):
    filled = int(rows[col].notna().sum())
    print('  %-14s %d/%d  (%.1f%%)' % (col, filled, len(rows), filled / n * 100))

print('=== errors seen ===')
errs = rows[rows.error != '']
display(errs[['filename', 'error']].head(20) if len(errs) else 'none')


## 8. Persist the output

Everything lands in `/kaggle/working`. **Save Version** to keep it, then attach that
output as an input to the next session so `hvac.db` is restored and finished files are
skipped.


In [ ]:
# Independent of cell 7 -- that cell may have been skipped or errored, and this
# one should still tell you what you have.
try:
    con.close()
except NameError:
    pass

for f in sorted(glob.glob('/kaggle/working/*.csv') + glob.glob('/kaggle/working/*.db')):
    print('%8.2f MB  %s' % (os.path.getsize(f) / 1e6, f))
print('Save Version to persist these, then attach this output next session.')
